# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [4]:
#Load the data set
aviation = pd.read_csv('AviationData.csv', encoding='latin1')

#Inspect datatypes and general info
aviation.info()


C:\Users\user\AppData\Local\Temp\ipykernel_24956\2048130451.py:2: DtypeWarning: Columns (6: Latitude, 7: Longitude, 28: Broad.phase.of.flight) have mixed types. Specify dtype option on import or set low_memory=False.
  aviation = pd.read_csv('AviationData.csv', encoding='latin1')


<class 'pandas.DataFrame'>
RangeIndex: 88889 entries, 0 to 88888
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Event.Id                88889 non-null  str    
 1   Investigation.Type      88889 non-null  str    
 2   Accident.Number         88889 non-null  str    
 3   Event.Date              88889 non-null  str    
 4   Location                88837 non-null  str    
 5   Country                 88663 non-null  str    
 6   Latitude                34382 non-null  object 
 7   Longitude               34373 non-null  object 
 8   Airport.Code            50132 non-null  str    
 9   Airport.Name            52704 non-null  str    
 10  Injury.Severity         87889 non-null  str    
 11  Aircraft.damage         85695 non-null  str    
 12  Aircraft.Category       32287 non-null  str    
 13  Registration.Number     87507 non-null  str    
 14  Make                    88826 non-null  str    
 

In [5]:
#Inspect missing values by column
aviation.isna().sum()

Event.Id                      0
Investigation.Type            0
Accident.Number               0
Event.Date                    0
Location                     52
Country                     226
Latitude                  54507
Longitude                 54516
Airport.Code              38757
Airport.Name              36185
Injury.Severity            1000
Aircraft.damage            3194
Aircraft.Category         56602
Registration.Number        1382
Make                         63
Model                        92
Amateur.Built               102
Number.of.Engines          6084
Engine.Type                7096
FAR.Description           56866
Schedule                  76307
Purpose.of.flight          6192
Air.carrier               72241
Total.Fatal.Injuries      11401
Total.Serious.Injuries    12510
Total.Minor.Injuries      11933
Total.Uninjured            5912
Weather.Condition          4492
Broad.phase.of.flight     27165
Report.Status              6384
Publication.Date          13771
dtype: i

In [6]:
#View summary statistics
aviation.describe()

,Number.of.Engines,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured
count,82805.000000,77488.000000,76379.000000,76956.000000,82977.000000
mean,1.146585,0.647855,0.279881,0.357061,5.325440
std,0.446510,5.485960,1.544084,2.235625,27.913634
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.000000,0.000000,0.000000,0.000000,0.000000
50%,1.000000,0.000000,0.000000,0.000000,1.000000
75%,1.000000,0.000000,0.000000,0.000000,2.000000
max,8.000000,349.000000,161.000000,380.000000,699.000000


## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [7]:
#Display all column names
print(aviation.columns)

Index(['Event.Id', 'Investigation.Type', 'Accident.Number', 'Event.Date',
       'Location', 'Country', 'Latitude', 'Longitude', 'Airport.Code',
       'Airport.Name', 'Injury.Severity', 'Aircraft.damage',
       'Aircraft.Category', 'Registration.Number', 'Make', 'Model',
       'Amateur.Built', 'Number.of.Engines', 'Engine.Type', 'FAR.Description',
       'Schedule', 'Purpose.of.flight', 'Air.carrier', 'Total.Fatal.Injuries',
       'Total.Serious.Injuries', 'Total.Minor.Injuries', 'Total.Uninjured',
       'Weather.Condition', 'Broad.phase.of.flight', 'Report.Status',
       'Publication.Date'],
      dtype='str')


In [54]:
aviation.iloc[52:59]

,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Airport.Code,Airport.Name,Injury.Severity,Aircraft.damage,...,Engine.Type,Purpose.of.flight,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date
52,20020917X01659,Accident,ATL82DA031,1982-01-07,"WESTMINISTER, MD",United States,2W2,CLEARVIEW,Non-Fatal,Substantial,...,Reciprocating,Personal,0.0,0.0,0.0,1.0,VMC,Approach,Probable Cause,07-01-1983
53,20020917X02141,Accident,FTW82FRG20,1982-01-07,"LAREDO, TX",United States,LRD,LAREDO INTL,Non-Fatal,Substantial,...,Reciprocating,Unknown,0.0,0.0,0.0,3.0,VMC,Standing,Probable Cause,07-01-1983
54,20020917X02414,Accident,MIA82FLD14,1982-01-08,"ODESSA, FL",United States,X41,W. PASCO,Fatal(1),Destroyed,...,Reciprocating,Personal,1.0,0.0,0.0,0.0,IMC,Approach,Probable Cause,08-01-1983
55,20020917X01881,Accident,CHI82FA024,1982-01-08,"NEW PHILADELPHI, OH",United States,PHO,HARRY CLEVER FIELD,Fatal(1),Destroyed,...,Reciprocating,Personal,1.0,1.0,0.0,0.0,VMC,Approach,Probable Cause,08-01-1983
56,20020917X02574,Accident,SEA82DA016,1982-01-08,"PULLMAN, WA",United States,PUW,PULLMAN-MOSCOW REGIONAL,Non-Fatal,Substantial,...,Reciprocating,Business,0.0,0.0,0.0,1.0,VMC,Landing,Probable Cause,08-01-1983
57,20020917X02484,Accident,NYC82DA019,1982-01-08,"CUTCHOGUE, NY",United States,NaN,NaN,Non-Fatal,Substantial,...,Reciprocating,Instructional,0.0,0.0,0.0,1.0,VMC,Cruise,Probable Cause,08-01-1983
58,20020917X02431,Accident,MKC82DA026,1982-01-08,"LIBERTY, MO",United States,NaN,MITHELL,Non-Fatal,Substantial,...,Reciprocating,Instructional,0.0,0.0,0.0,2.0,VMC,Landing,Probable Cause,08-01-1983


In [18]:
print(aviation['Purpose.of.flight'].unique())

<StringArray>
[                 'Personal',                         nan,
                  'Business',             'Instructional',
                   'Unknown',                     'Ferry',
       'Executive/corporate',        'Aerial Observation',
        'Aerial Application',           'Public Aircraft',
                 'Skydiving',            'Other Work Use',
               'Positioning',               'Flight Test',
             'Air Race/show',                  'Air Drop',
 'Public Aircraft - Federal',                'Glider Tow',
   'Public Aircraft - Local',             'External Load',
   'Public Aircraft - State',                'Banner Tow',
              'Firefighting',             'Air Race show',
                      'PUBS',                      'ASHO',
                      'PUBL']
Length: 27, dtype: str


In [33]:
print(aviation['Aircraft.damage'].unique())

<StringArray>
['Destroyed', 'Substantial', 'Minor', nan, 'Unknown']
Length: 5, dtype: str


In [ ]:
#Fill out the null with unknown
cat_col = ['Aircraft.Category', 'Make', 'Model' , 'Engine.Type', 'Purpose.of.flight']
for col in cat_col:
    aviation[col] = aviation[col].fillna('Unknown')

In [45]:
aviation.loc[0:20,'Aircraft.Category']

0        Unknown
1        Unknown
2        Unknown
3        Unknown
4        Unknown
5       Airplane
6        Unknown
7       Airplane
8       Airplane
9        Unknown
10       Unknown
11       Unknown
12      Airplane
13      Airplane
14      Airplane
15      Airplane
16    Helicopter
17      Airplane
18      Airplane
19    Helicopter
20      Airplane
Name: Aircraft.Category, dtype: str

In [46]:
#Filling the null in injury columns with 0
injury_cols = ['Total.Fatal.Injuries','Total.Serious.Injuries', 'Total.Minor.Injuries', 'Total.Uninjured']
for col in injury_cols:
    aviation[col] = aviation[col].fillna(0)

In [47]:
aviation.loc[0:20,'Total.Fatal.Injuries']

0     2.0
1     4.0
2     3.0
3     2.0
4     1.0
5     0.0
6     4.0
7     0.0
8     0.0
9     0.0
10    0.0
11    0.0
12    0.0
13    1.0
14    1.0
15    2.0
16    0.0
17    3.0
18    0.0
19    0.0
20    0.0
Name: Total.Fatal.Injuries, dtype: float64

In [ ]:
columns_to_drop = ['Latitude', 'Longitude', 'FAR.Description','Schedule','Air.carrier']

aviation = aviation.drop(columns=columns_to_drop)

print(aviation.columns.tolist())

KeyError: "['Latitude', 'Longitude', 'FAR.Description', 'Schedule', 'Air.carrier'] not found in axis"

In [60]:
aviation.iloc[0:6]

,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Airport.Code,Airport.Name,Injury.Severity,Aircraft.damage,...,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date,Total.On.Board,Severe.Injury.Rate
0,20001218X45444,Accident,SEA87LA080,1948-10-24,"MOOSE CREEK, ID",United States,NaN,NaN,Fatal(2),Destroyed,...,2.0,0.0,0.0,0.0,UNK,Cruise,Probable Cause,NaN,2.0,1.0
1,20001218X45447,Accident,LAX94LA336,1962-07-19,"BRIDGEPORT, CA",United States,NaN,NaN,Fatal(4),Destroyed,...,4.0,0.0,0.0,0.0,UNK,Unknown,Probable Cause,19-09-1996,4.0,1.0
2,20061025X01555,Accident,NYC07LA005,1974-08-30,"Saltville, VA",United States,NaN,NaN,Fatal(3),Destroyed,...,3.0,0.0,0.0,0.0,IMC,Cruise,Probable Cause,26-02-2007,3.0,1.0
3,20001218X45448,Accident,LAX96LA321,1977-06-19,"EUREKA, CA",United States,NaN,NaN,Fatal(2),Destroyed,...,2.0,0.0,0.0,0.0,IMC,Cruise,Probable Cause,12-09-2000,2.0,1.0
4,20041105X01764,Accident,CHI79FA064,1979-08-02,"Canton, OH",United States,NaN,NaN,Fatal(1),Destroyed,...,1.0,2.0,0.0,0.0,VMC,Approach,Probable Cause,16-04-1980,3.0,1.0
5,20170710X52551,Accident,NYC79AA106,1979-09-17,"BOSTON, MA",United States,NaN,NaN,Non-Fatal,Substantial,...,0.0,0.0,1.0,44.0,VMC,Climb,Probable Cause,19-09-2017,45.0,0.0


### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

In [55]:
# Estimate the total number of passengers/crew by summing all injury statuses and uninjured individuals.
aviation['Total.On.Board'] = (
    aviation['Total.Fatal.Injuries'] + 
    aviation['Total.Serious.Injuries'] + 
    aviation['Total.Minor.Injuries'] + 
    aviation['Total.Uninjured']
)

# calculates the fraction of people on board who suffered serious or fatal injuries.
# We use a conditional safe division to handle cases where Total.On.Board is 0 (to avoid zero division errors).
aviation['Severe.Injury.Rate'] = ((aviation['Total.Fatal.Injuries'] + aviation['Total.Serious.Injuries']) / aviation['Total.On.Board']).fillna(0)

# Preview the results
print(aviation[['Total.On.Board', 'Severe.Injury.Rate']].describe())

       Total.On.Board  Severe.Injury.Rate
count    88889.000000        88889.000000
mean         6.085624            0.285751
std         27.845104            0.435478
min          0.000000            0.000000
25%          1.000000            0.000000
50%          2.000000            0.000000
75%          2.000000            1.000000
max        699.000000            1.000000


**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [61]:
# Standardize the 'Aircraft.Damage' text by trimming spaces and making it title case. 
# Fill missing damage descriptions with 'Unknown'.
aviation['Aircraft.damage'] = aviation['Aircraft.damage'].str.strip().str.title()
aviation['Aircraft.damage'] = aviation['Aircraft.damage'].fillna('Unknown')

# Print unique values to confirm structural categories (e.g., 'Destroyed', 'Substantial', 'Minor')
print("Unique damage categories:", aviation['Aircraft.damage'].unique())

# --- DERIVED COLUMN 3: Is Destroyed (Binary Indicator) ---
# Creates a boolean/integer column (1 for true, 0 for false) tracking if the aircraft was completely destroyed.
aviation['Is.Destroyed'] = (aviation['Aircraft.damage'] == 'Destroyed').astype(int)

# Verify the proportions of destroyed aircraft
print("\nProportion of aircraft destroyed:")
print(aviation['Is.Destroyed'].value_counts(normalize=True))

Unique damage categories: <StringArray>
['Destroyed', 'Substantial', 'Minor', 'Unknown']
Length: 4, dtype: str

Proportion of aircraft destroyed:
Is.Destroyed
0    0.790492
1    0.209508
Name: proportion, dtype: float64


### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

In [62]:
# Clean the text formatting
aviation['Make'] = aviation['Make'].str.strip().str.upper()
aviation['Make'] = aviation['Make'].fillna('UNKNOWN')
make_counts = aviation['Make'].value_counts()

# Filter to keep only Makes with at least 50 entries
valid_makes = make_counts[make_counts >= 50].index
aviation = aviation[aviation['Make'].isin(valid_makes)].copy()

# Print out the remaining top unique makes to verify the cleanup
print(f"Number of unique makes remaining: {aviation['Make'].nunique()}")
print("\nTop 10 manufacturers in the dataset:")
print(aviation['Make'].value_counts().head(10))

Number of unique makes remaining: 99

Top 10 manufacturers in the dataset:
Make
CESSNA      27149
PIPER       14870
BEECH        5372
BOEING       2745
BELL         2722
MOONEY       1334
ROBINSON     1230
GRUMMAN      1172
BELLANCA     1045
HUGHES        932
Name: count, dtype: int64


### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [ ]:
# Clean the Model column strings and remove any remaining NaNs
aviation['Model'] = aviation['Model'].str.strip().str.upper()
aviation = aviation.dropna(subset=['Model'])

# Group by 'Model' and see how many unique 'Make' values each model maps to
models_with_multiple_makes = aviation.groupby('Model')['Make'].nunique()
shared_models = models_with_multiple_makes[models_with_multiple_makes > 1]

print(f"Number of model names shared by more than one manufacturer: {len(shared_models)}")
if len(shared_models) > 0:
    print("\nExample of non-unique model strings across manufacturers:")
    sample_model = shared_models.index[0]
    print(aviation[aviation['Model'] == sample_model][['Make', 'Model']].drop_duplicates().head())

# Combining Make and Model
aviation['Plane.Type'] = aviation['Make'] + " " + aviation['Model']

print("\nColumn 'Plane.Type' successfully created.")
print(aviation[['Make', 'Model', 'Plane.Type']].head())

Number of model names shared by more than one manufacturer: 503

Example of non-unique model strings across manufacturers:
                 Make Model
714    AERO COMMANDER   100
3479            BEECH   100
14526        ROCKWELL   100
32351          FOKKER   100

Derived column 'Plane.Type' successfully created.
       Make     Model      Plane.Type
0   STINSON     108-3   STINSON 108-3
1     PIPER  PA24-180  PIPER PA24-180
2    CESSNA      172M     CESSNA 172M
3  ROCKWELL       112    ROCKWELL 112
4    CESSNA       501      CESSNA 501


### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [64]:
# List of columns to clean
target_columns = ['Engine.Type', 'Weather.Condition', 'Purpose.of.flight','Broad.phase.of.flight']

# --- 1. INSPECT INITIAL UNIQUE VALUES ---
print("BEFORE CLEANING (Unique Values Sample)")
for col in target_columns:
    print(f"\n{col} unique values (first 10):")
    print(aviation[col].unique()[:10])

for col in target_columns:
    aviation[col] = aviation[col].astype(str).str.strip().str.upper()
    
    aviation[col] = aviation[col].replace(['NAN', 'UNK', 'NONE', ''], 'UNKNOWN')

aviation['Number.of.Engines'] = pd.to_numeric(aviation['Number.of.Engines'], errors='coerce')

print("\n" + "="*40 + "\nAFTER CLEANING ")
for col in target_columns:
    print(f"\nCleaned {col} unique values:")
    print(aviation[col].unique())

print("\nValue counts for Number.of.Engines:")
print(aviation['Number.of.Engines'].value_counts(dropna=False))

BEFORE CLEANING (Unique Values Sample)

Engine.Type unique values (first 10):
<StringArray>
[  'Reciprocating',         'Unknown',       'Turbo Fan',     'Turbo Shaft',
      'Turbo Prop',       'Turbo Jet', 'Geared Turbofan',              'LR',
             'UNK',            'NONE']
Length: 10, dtype: str

Weather.Condition unique values (first 10):
<StringArray>
['UNK', 'IMC', 'VMC', nan, 'Unk']
Length: 5, dtype: str

Purpose.of.flight unique values (first 10):
<StringArray>
[           'Personal',             'Unknown',            'Business',
       'Instructional',               'Ferry', 'Executive/corporate',
  'Aerial Observation',  'Aerial Application',     'Public Aircraft',
           'Skydiving']
Length: 10, dtype: str

Broad.phase.of.flight unique values (first 10):
<StringArray>
[     'Cruise',     'Unknown',    'Approach',       'Climb',     'Takeoff',
     'Landing',        'Taxi',     'Descent', 'Maneuvering',   'Go-around']
Length: 10, dtype: str

AFTER CLEANING 

Clean

### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [65]:
# Save the fully cleaned dataset to a new CSV file
output_filename = 'aviation_cleaned.csv'
aviation.to_csv(output_filename, index=False)

print(f"Success! Cleaned dataset saved to: '{output_filename}'")
print(f"Final shape of saved dataset: {aviation.shape}")

Success! Cleaned dataset saved to: 'aviation_cleaned.csv'
Final shape of saved dataset: (74739, 30)
